In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder

# Set visual style for plots
sns.set_theme(style="whitegrid")

def load_and_merge_data():
    """
    Loads F1 datasets and merges them into a single dataframe.
    Assumes standard Kaggle filenames.
    """
    # Load the core CSVs
    # The CSVs used are downloaded from a Kaggle dataset named "Formula 1 World Championship (1950 - 2024)" by Vopani
    results = pd.read_csv('datasets/results.csv')
    races = pd.read_csv('datasets/races.csv')
    drivers = pd.read_csv('datasets/drivers.csv')
    constructors = pd.read_csv('datasets/constructors.csv')

    # Merge 'races' with 'results' to get the year and circuit info for each result
    # We rename 'name' in races to 'GP_Name' to avoid confusion with driver names
    races = races.rename(columns={'name': 'GP_Name', 'date': 'race_date'})
    df = pd.merge(results, races[['raceId', 'year', 'GP_Name', 'circuitId']], on='raceId', how='left')

    # Merge with 'drivers' to get driver names
    # Only keep relevant columns
    df = pd.merge(df, drivers[['driverId', 'driverRef', 'nationality']], on='driverId', how='left')

    # Merge with 'constructors' to get team names
    constructors = constructors.rename(columns={'name': 'team_name', 'nationality': 'team_nationality'})
    df = pd.merge(df, constructors[['constructorId', 'team_name', 'team_nationality']], on='constructorId', how='left')

    return df

# Load the raw dataframe
df = load_and_merge_data()
print(f"Total raw rows: {len(df)}")
df.head()

In [ ]:
def preprocess_data(df):
    """
    Cleans data and creates the Target Variable for classification.
    """
    # F1 points systems and reliability changed drastically over time.
    # Focusing on 2010-2024 makes the model more consistent as this is the newest points system.
    df_modern = df[df['year'] >= 2010].copy()

    # Create the Target Variable: "Podium Finish"
    # We want to predict if a driver finishes in the Top 3.
    # Logic: 1 if positionOrder <= 3, else 0.
    df_modern['is_podium'] = df_modern['positionOrder'].apply(lambda x: 1 if x <= 3 else 0)

    # Numeric features we keep as-is
    numeric_features = ['grid', 'points']
    # Categorical features to encode
    categorical_features = ['team_name']
    # sparse_output=False: Returns a numpy array. setting to true will return a sparse matrix where (x,y)=result. Row x, Col y is result (0 or 1)
    # handle_unknown='ignore': When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. 
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded_array = encoder.fit_transform(df_modern[categorical_features])

    # get_feature_names_out() automatically names columns from encoding 
    # Will add more featuers in the future
    encoded_df = pd.DataFrame(
        encoded_array,
        columns=encoder.get_feature_names_out(categorical_features),
        index=df_modern.index   # Keep the same rows as df_modern
    )
    
    final_df = pd.concat([df_modern[numeric_features], encoded_df, df_modern['is_podium']], axis=1)
    
    return final_df, encoder

cleaned_df, trained_encoder = preprocess_data(df)
cleaned_df.head()   

In [ ]:
# Fitting the model
# Baseline will use logistic regression and compare its performance with a support vector machine (SVM)
# Implement anything that you guys think will perform better


In [ ]:
# Plotting stats and results (e.g. logistic vs svm)
# Evalutions